# Read and write a benchmark
Use the competition environment and **Run All**. `CommonRoadFileReader.open()` returns the road/traffic scenario and its planning problems. This example reads the public Augsburg starter.


In [ ]:
from commonroad.common.file_reader import CommonRoadFileReader
from planner.common.scenarios import ASSETS

path = ASSETS / "scenarios/public/DEU_AugsburgCRC26P-23_16300_I-1.xml"
scenario, problems = CommonRoadFileReader(str(path)).open()
problem = next(iter(problems.planning_problem_dict.values()))
print(scenario.scenario_id, "dt:", scenario.dt)
print("Initial state:", problem.initial_state)
print("Goal:", problem.goal)


## Understand the task
The initial state gives position, heading, speed and time. Each goal state specifies an arrival window, a spatial region and any additional heading/speed constraints. Entering a qualifying region during its window is enough; the vehicle need not remain there until the deadline.

Traffic trajectories in the source XML guide SUMO's routes. They are not future observations available to your planner. SUMO determines actual motion during a run.


In [ ]:
{
    "lanelets": len(scenario.lanelet_network.lanelets),
    "moving_actors": len(scenario.dynamic_obstacles),
    "static_obstacles": len(scenario.static_obstacles),
    "goal_alternatives": len(problem.goal.state_list),
}


## Write a copy
Use a separate path for experiments; keep competition input files unchanged. This temporary copy demonstrates XML I/O. For an executed drive and its simulated traffic, use `export_result()` as shown in [04](04_reuse_and_evaluate_planners.ipynb).


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from commonroad.common.file_writer import CommonRoadFileWriter, OverwriteExistingFile

with TemporaryDirectory(prefix="commonroad-xml-example-") as directory:
    copy_path = Path(directory) / path.name
    writer = CommonRoadFileWriter(scenario, problems, decimal_precision=16)
    writer.write_to_file(
        str(copy_path),
        overwrite_existing_file=OverwriteExistingFile.ALWAYS,
        check_validity=True,
    )
    restored, restored_problems = CommonRoadFileReader(str(copy_path)).open()
print(restored.scenario_id, list(restored_problems.planning_problem_dict))
